# Mellea for GRC: Generative Computing for Compliance Automation

This tutorial introduces the basics of Generative Computing using Mellea, with a focus on **Governance, Risk, and Compliance (GRC)** workflows for Technology/SaaS companies.

During this tutorial, we will:
1. Get up and running with Mellea for GRC tasks
2. See the Instruct-Validate-Repair pattern for policy compliance validation
3. Build composable compliance assessment libraries using the `@generative` decorator
4. Process compliance documents with Mellea and Docling
5. Create policy-as-code systems using Mellea's `MObject` protocol

**Target Audience**: GRC professionals, compliance engineers, security teams, and developers building compliance automation for SaaS companies (SOC2, ISO 27001, etc.)

## Getting Started

Run the first cell during our introduction. The first cell will:
 * Download and install ollama on your Colab instance
 * Download the `granite3.3:8b` model weights
 * Install Mellea and dependencies

In [ ]:
# Install ollama.
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
!nohup ollama serve >/dev/null 2>&1 &

# Download the granite:3.3:8b weights.
!ollama pull granite3.3:8b
!ollama pull llama3.2:3b

# install Mellea.
!uv pip install "mellea[all] @ git+https://github.com/generative-computing/mellea@v0.0.5"

# Run docling once to download model weights.
from mellea.stdlib.docs.richdocument import RichDocument

RichDocument.from_document_file("https://mellea.ai")

# Some UI niceness.
from IPython.display import HTML, display  # noqa: E402


def set_css():
    display(HTML("\n<style>\n pre{\n white-space: pre-wrap;\n}\n</style>\n"))


get_ipython().events.register("pre_run_cell", set_css)

# Lab 1: Hello, Mellea - Compliance Checking Basics

Running `mellea.start_session()` initializes a new `MelleaSession`. The session holds three things:
1. The model to use for this session. In this tutorial we will use granite3.3:8b.
2. An inference engine; i.e., the code that actually calls our model. We will be using ollama, but you can also use Huggingface or any OpenAI-compatible endpoint.
3. A `Context`, which tells Mellea how to remember context between requests. This is sometimes called the "Message History" in other frameworks. Throughout this tutorial, we will be using a `SimpleContext`. In `SimpleContext`s, **every request starts with a fresh context**. There is no preserved chat history between requests. Mellea provides other types of context, but today we will not be using those features. See the Tutorial for further details.

In this lab, we'll explore how Mellea can help with basic GRC (Governance, Risk, and Compliance) tasks like understanding compliance requirements and generating policy statements.

In [ ]:
import mellea

m = mellea.start_session()

# Example 1: Understanding compliance requirements
answer = m.chat(
    "What are the key requirements of SOC2 CC6.1 (Logical and Physical Access Controls)? Provide a brief summary."
)
print("=== SOC2 CC6.1 Requirements ===")
print(answer.content)

# Example 2: Generate a basic access control policy statement
print("\n=== Generated Access Control Policy ===")
policy = m.instruct(
    "Write a brief access control policy statement for a SaaS company that addresses user authentication requirements, including MFA and password policies."
)
print(policy.value)

# Lab 2: Instruct-Validate-Repair - Password Policy Compliance

Instruct-Validate-Repair is a design pattern for building robust automation using LLMs. The idea is simple:
1. Instruct the model to perform a task and specify requirements on the output of the task.
2. Validate that these requirements are satisfied by the model's output.
3. If any requirements fail, try to repair.

In this lab, we'll use IVR to generate a password policy that complies with SOC2 requirements. We'll mix LLM-based validation (using the model as a judge) with programmatic validation (using Python code) to ensure the generated policy meets all our compliance requirements.

In [ ]:
import mellea
from mellea.stdlib.requirement import check, req, simple_validate
from mellea.stdlib.sampling import RejectionSamplingStrategy

requirements = [
    req("The policy must specify a minimum password length of at least 8 characters"),  # r1 - LLM judge
    req(
        "The policy must mention multi-factor authentication or MFA",
        validation_fn=simple_validate(
            lambda x: "multi-factor" in x.lower() or "mfa" in x.lower()
        ),
    ),  # r2 - programmatic validation
    req(
        "The policy must prohibit password reuse for at least 5 previous passwords",
        validation_fn=simple_validate(
            lambda x: any(str(i) in x for i in range(5, 25))  # looks for numbers 5-24
        ),
    ),  # r3 - programmatic validation
    check("Do not use overly technical jargon that non-technical employees wouldn't understand"),  # r4 - check only (not in instruct)
]


def generate_password_policy(m: mellea.MelleaSession, company_name: str) -> str:
    policy_candidate = m.instruct(
        "Write a password policy for {{company_name}}, a SaaS company, that complies with SOC2 requirements for access control.",
        requirements=requirements,
        strategy=RejectionSamplingStrategy(loop_budget=5),
        user_variables={"company_name": company_name},
        return_sampling_results=True,
    )
    if policy_candidate.success:
        return str(policy_candidate.result)
    else:
        print("Warning: Could not generate fully compliant policy. Using best attempt.")
        return policy_candidate.sample_generations[0].value


m = mellea.start_session()
print("=== Generated Password Policy ===")
print(
    generate_password_policy(m, "Acme SaaS Corp")
)

## Lab 3: Generative Slots - Compliance Assessment Functions

In classical programming, pure (stateless) functions are a simple and powerful abstraction. A pure function takes inputs, computes outputs, and has no side effects. Generative programs can also use functions as abstraction boundaries, but in a generative program the meaning of the function can be given by an LLM instead of an interpreter or compiler. This is the idea behind a GenerativeSlot.

A GenerativeSlot is a function whose implementation is provided by an LLM. In Mellea, you define these using the `@generative` decorator. The function signature specifies the interface, and the docstring (or type annotations) guide the LLM in producing the output. 

In this lab, we'll build a library of compliance assessment functions that can classify evidence, assess controls, and generate audit findings. We'll then see how to compose these functions safely using **compositionality checks** (contracts).

In [ ]:
from mellea import generative
from typing import Literal
import mellea

m = mellea.start_session()

@generative
def classify_evidence_type(evidence_text: str) -> Literal["screenshot", "log_file", "policy_document", "attestation", "other"]:
    """Classify what type of compliance evidence this text represents."""

# Example: Classify different types of evidence
evidence_sample = "Screenshot of Okta admin panel showing MFA enforcement policy enabled for all users."
evidence_type = classify_evidence_type(m, evidence_text=evidence_sample)
print(f"Evidence Type: {evidence_type}")

Let's see how **compositionality checks** can be used to safely combine libraries of generative functions. We'll build two independent libraries:
1. **Control Evidence Analyzers** - Extract and classify compliance evidence
2. **Control Assessment Functions** - Evaluate control effectiveness

Then we'll use compositionality checks to ensure outputs from analyzers are suitable inputs for assessments.

In [ ]:
from mellea import generative

################################################################################
# CONTROL EVIDENCE ANALYZER LIBRARY                                            #
################################################################################

@generative
def classify_evidence_type(evidence_text: str) -> Literal["screenshot", "log_file", "policy_document", "attestation", "other"]:
    """Classify what type of compliance evidence this text represents."""

@generative
def extract_control_reference(evidence_text: str) -> str:
    """Extract which SOC2 control (e.g., CC6.1, CC7.2) this evidence relates to. Return 'Unknown' if no control is referenced."""

@generative
def assess_evidence_completeness(evidence_text: str) -> Literal["complete", "incomplete", "unclear"]:
    """Determine if the evidence contains sufficient detail to demonstrate control operation."""


################################################################################
# CONTROL ASSESSMENT LIBRARY                                                   #
################################################################################

@generative
def evaluate_control_design(evidence_summary: str) -> Literal["effective", "ineffective", "needs_improvement"]:
    """Given evidence that a control exists, evaluate if the control design is effective."""

@generative
def evaluate_control_operation(evidence_summary: str) -> Literal["operating_effectively", "not_operating_effectively", "insufficient_evidence"]:
    """Given evidence of control operation, evaluate if it's operating as designed."""

@generative
def generate_audit_finding(evidence_summary: str, design_rating: str, operation_rating: str) -> str:
    """Generate a formal audit finding based on control assessment results."""


################################################################################
# COMPOSITIONALITY CHECKS                                                      #
################################################################################

@generative
def has_clear_control_reference(text: str) -> Literal["yes", "no"]:
    """Check if the text clearly identifies which control is being assessed."""

@generative
def contains_sufficient_detail(text: str) -> Literal["yes", "no"]:
    """Check if the text has enough detail to support a control assessment."""

@generative
def has_design_and_operation_conclusions(text: str) -> Literal["yes", "no"]:
    """Check if the text contains both design and operating effectiveness conclusions."""

### Access Control Evidence Assessment

Let's use the evidence analyzer library to assess compliance evidence for an access control system.

In [ ]:
################################################################################
# Sample Evidence: Access Control Implementation (SOC2 CC6.1)                  #
################################################################################

evidence = """
During our review period (January 1 - December 31, 2024), we examined the company's 
user access provisioning process for the production environment to assess compliance 
with SOC2 CC6.1 (Logical and Physical Access Controls).

Evidence reviewed for CC6.1:
- Screenshot of Okta admin panel showing MFA enforcement policy enabled for all users
- Azure AD conditional access policies requiring MFA for privileged roles
- Sample of 10 new hire onboarding tickets showing access request approval workflow
- Quarterly access reviews from Q1, Q2, Q3, Q4 2024 showing 95%+ completion rates
- Evidence of 3 access terminations processed within 24 hours of employee departure

The control operates as follows: When a new employee joins, their manager submits 
an access request ticket in Jira. IT Security reviews the request against the 
role-based access control (RBAC) matrix. Upon approval, access is provisioned in 
Okta with MFA required for all accounts. Access permissions are reviewed quarterly 
by managers and IT Security. Upon employee departure, HR triggers an automated 
workflow that revokes all access within 24 hours.

Control testing results: We tested 25 access provisioning events and found all 
followed the documented process. All 4 quarterly access reviews were completed 
on time. All 3 terminations were processed within the 24-hour requirement.
"""

print("=== Step 1: Evidence Analysis ===")
evidence_type = classify_evidence_type(m, evidence_text=evidence)
control_ref = extract_control_reference(m, evidence_text=evidence)
completeness = assess_evidence_completeness(m, evidence_text=evidence)

print(f"Evidence Type: {evidence_type}")
print(f"Control Reference: {control_ref}")
print(f"Completeness Assessment: {completeness}")

### Composing Evidence Analysis with Control Assessment

Now let's use compositionality checks to ensure the evidence meets preconditions before performing the control assessment.

In [ ]:
################################################################################
# COMPOSITIONALITY WORKFLOW: Safe Evidence → Assessment Pipeline              #
################################################################################

print("\n=== Step 2: Compositionality Checks ===")
# Only proceed with assessment if evidence meets preconditions
if has_clear_control_reference(m, text=evidence) == "yes":
    if contains_sufficient_detail(m, text=evidence) == "yes":
        print("✓ Evidence meets preconditions for assessment")
        
        print("\n=== Step 3: Control Assessment ===")
        design = evaluate_control_design(m, evidence_summary=evidence)
        operation = evaluate_control_operation(m, evidence_summary=evidence)
        
        print(f"Design Effectiveness: {design}")
        print(f"Operating Effectiveness: {operation}")
        
        # Only generate finding if we have both assessments
        assessment_summary = f"Design: {design}, Operation: {operation}"
        if has_design_and_operation_conclusions(m, text=assessment_summary) == "yes":
            print("\n=== Step 4: Generate Audit Finding ===")
            finding = generate_audit_finding(
                m, 
                evidence_summary=evidence,
                design_rating=design,
                operation_rating=operation
            )
            print(f"Finding:\n{finding}")
        else:
            print("✗ Insufficient conclusions to generate finding")
    else:
        print("✗ Evidence lacks sufficient detail for assessment")
else:
    print("✗ Evidence does not clearly reference a control")

In [ ]:
# The compositionality checks ensure we only proceed with assessment when the evidence
# meets our quality standards. This prevents generating unreliable audit findings from
# incomplete or unclear evidence, demonstrating how to build robust compliance workflows.

# LAB 4: Docling and Mellea - Compliance Document Processing

In this lab, we will use both Docling and Mellea to extract and analyze data from compliance documents. This is useful for processing SOC2 reports, control matrices, audit findings, and other compliance artifacts.

We'll demonstrate how to extract tables from documents and use Mellea to augment them with gap analysis and risk assessments.

In [ ]:
from mellea.stdlib.docs.richdocument import RichDocument

# In a real GRC workflow, this would be a SOC2 report, control matrix, or audit document
# For this tutorial, we'll use a research paper as an example
rd = RichDocument.from_document_file("https://arxiv.org/pdf/1906.04043")

## Extract Control Matrix from Document

We can use Docling to extract tables from compliance documents. In a real scenario, this might be a table of controls, audit findings, or risk assessments.

In [ ]:
from mellea.stdlib.docs.richdocument import Table

# Extract the first table from the document
# In a real GRC scenario, this would be a control matrix or audit findings table
table1: Table = rd.get_tables()[0]
print("=== Original Table from Document ===")
print(table1.to_markdown())

# Lab 4: Augmenting Tables with Compliance Analysis

The Table object is Mellea-ready and can be transformed using LLMs. In this example, we'll add compliance-related columns to demonstrate how you might augment a control matrix with gap analysis or risk assessments.

In [ ]:
from mellea.backends.model_ids import META_LLAMA_3_2_3B
from mellea.backends.ollama import OllamaModelBackend
from mellea.backends.types import ModelOption
import mellea

# Create a new session (you can also use different models for different tasks)
m_transform = mellea.MelleaSession(backend=OllamaModelBackend(model_id=META_LLAMA_3_2_3B))

print("\n=== Transformed Table with Compliance Assessment ===")

# Try multiple seeds to get reliable parsing
# In a real GRC workflow, you might add columns like:
# - "Compliance Status" (Implemented/Partial/Not Implemented)
# - "Gap Analysis" (what's missing)
# - "Risk Level" (High/Medium/Low)
# - "Recommendation" (what to do next)

for seed in [x * 12 for x in range(5)]:
    table2 = m_transform.transform(
        table1,
        "Add a 'Compliance Relevance' column that identifies if this relates to security, privacy, or compliance frameworks. Then add a 'Risk Assessment' column that evaluates potential compliance risks.",
        model_options={ModelOption.SEED: seed},
    )
    if isinstance(table2, Table):
        print(table2.to_markdown())
        
        # Query the transformed table
        print("\n=== Compliance Risk Query ===")
        risk_query = m_transform.query(
            table2, 
            "Based on this analysis, which items pose the highest compliance risk?"
        )
        print(risk_query.content)
        break
    else:
        print("==== Retrying table transformation... ====")

The model has transformed the table and added compliance-relevant analysis. You can now use `m.query(table2, "question")` to ask questions about the compliance implications, or continue transforming with `m.transform(table2, "instruction")` to further augment the analysis.

In real GRC workflows, you might process:
- SOC2 audit reports to extract control effectiveness
- Risk registers to add mitigation recommendations
- Compliance matrices to identify gaps
- Vendor security questionnaires to assess third-party risk


# Lab 5: MObjects - Compliance Control Database (Policy-as-Code)

Object-oriented programming (OOP) is a powerful paradigm for organizing code: you group related data and the methods that operate on that data into classes. In the world of LLMs, a similar organizational principle emerges—especially when you want to combine structured data with LLM-powered "tools" or operations. This is where Mellea's MObject abstraction comes in.

**The MObject Pattern**: You should store data alongside its relevant operations (tools). This allows LLMs to interact with both the data and methods in a unified, structured manner. It also simplifies the process of exposing only the specific fields and methods you want the LLM to access.

The MObject pattern also provides a way of evolving existing classical codebases into generative programs. Mellea's @mify decorator lets you turn any class into an MObject.

In this lab, we'll create a **ComplianceControlDatabase** that demonstrates **policy-as-code**: compliance controls stored as version-controlled data structures that can be queried and updated programmatically. This enables GitOps workflows for GRC.

In [ ]:
from io import StringIO
import pandas as pd
from mellea.stdlib.mify import mify


@mify(fields_include={"controls_table"}, template="{{ controls_table }}")
class ComplianceControlDatabase:
    controls_table: str = """| Control_ID | Category | Status | Last_Review | Owner |
| ---------- | -------- | ------ | ----------- | ----- |
| CC6.1 | Logical Access | Implemented | 2024-12-01 | IT Security |
| CC6.2 | Change Management | Implemented | 2024-11-15 | DevOps |
| CC6.3 | System Operations | Partial | 2024-10-20 | SRE Team |
| CC7.1 | Threat Detection | Not Implemented | 2024-09-30 | Security Ops |
| CC7.2 | Incident Response | Implemented | 2024-12-10 | Incident Response |"""

    def __init__(self, *, controls_table: str | None = None):
        if controls_table is not None:
            self.controls_table = controls_table

    def _parse_table(self, table: str) -> pd.DataFrame:
        """Internal method to parse markdown table to DataFrame"""
        # Clean up the markdown table - drop separator row and strip whitespace/pipes
        lines = [
            line.strip()
            for line in table.strip().splitlines()
            if not set(line.strip()) <= {"|", "-", " "}
        ]
        cleaned = "\n".join(lines)

        # Read into dataframe
        df = pd.read_csv(StringIO(cleaned), sep="|")
        df = df.rename(columns=lambda x: x.strip())  # strip spaces from column names
        df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
        return df

    def update_control_status(self, control_id: str, new_status: str) -> "ComplianceControlDatabase":
        """Update the status of a specific control (Implemented/Partial/Not Implemented)"""
        df = self._parse_table(self.controls_table)
        df.loc[df["Control_ID"] == control_id, "Status"] = new_status
        return ComplianceControlDatabase(
            controls_table=df.to_csv(sep="|", index=False, header=True)
        )

    def update_review_date(self, control_id: str, review_date: str) -> "ComplianceControlDatabase":
        """Update the last review date for a control"""
        df = self._parse_table(self.controls_table)
        df.loc[df["Control_ID"] == control_id, "Last_Review"] = review_date
        return ComplianceControlDatabase(
            controls_table=df.to_csv(sep="|", index=False, header=True)
        )

    def get_controls_by_status(self, status: str) -> pd.DataFrame:
        """Get all controls with a specific status"""
        df = self._parse_table(self.controls_table)
        return df[df["Status"] == status]

In [ ]:
# Initialize the compliance control database
db = ComplianceControlDatabase()
m = mellea.start_session()

print("=== Initial Compliance Control Database ===")
print(db.controls_table)

# Example 1: Query the database using LLM
print("\n=== Query: Which controls need attention? ===")
query_result = m.query(
    db, 
    "Which controls are not fully implemented and should be prioritized?"
)
print(query_result.content)

# Example 2: Transform using natural language
print("\n=== Transform: Updating CC7.1 status ===")
db = m.transform(
    db,
    "Update the status of CC7.1 (Threat Detection) to 'Implemented' and set the last review date to 2024-12-29"
)
print(db.controls_table)

# Example 3: Query the updated database
print("\n=== Query: Current compliance posture ===")
posture = m.query(
    db,
    "What is our current compliance posture? How many controls are fully implemented versus partial or not implemented?"
)
print(posture.content)

# Example 4: Direct method call (programmatic update)
print("\n=== Programmatic Update: CC6.3 status ===")
db = db.update_control_status("CC6.3", "Implemented")
db = db.update_review_date("CC6.3", "2024-12-29")
print(db.controls_table)

# Example 5: Generate a compliance summary report
print("\n=== Generate Compliance Summary Report ===")
summary = m.query(
    db,
    "Generate a brief executive summary of our SOC2 control implementation status, highlighting any gaps and overall readiness."
)
print(summary.content)

print("\n=== Policy-as-Code Benefits ===")
print("This ComplianceControlDatabase can be:")
print("- Version controlled in Git (audit trail of all changes)")
print("- Updated via CI/CD pipelines (GitOps)")
print("- Queried programmatically for compliance dashboards")
print("- Transformed using natural language (accessible to non-technical users)")
print("- Integrated into automated compliance workflows")